<a href="https://colab.research.google.com/github/jwy0227-gif/Document-QA-RAG-System-FastAPI/blob/main/PA4_CSE151.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PA4: Preparing and Debugging Language-Model Inputs
CSE 151B | Fall 2026  
Due date: **November 2nd, 2026**

Prepare requests and reference stories for [TinyStories-1M](https://huggingface.co/roneneldan/TinyStories-1M), a small story-writing model. Then find inputs that expose errors in three supplied functions.

- **Part 1 (50 points):** tokenize examples, create batches, and compute loss.
- **Part 2 (50 points):** demonstrate and repair three bugs.

Complete the TODOs. The **prompt** is a request to use three words. The **response** is a reference story, followed by an end-of-sequence (**EOS**) token. The model sees the prompt, but the loss should score only predictions of the response and EOS.

**Make sure your notebook is submitted with all code and outputs visible, and your written responses should fall within designated boundaries to be processed.**


**Study overview:** TinyStories-1M is a small causal language model. Given a request such as “Write a story using moon, fox, boat,” it predicts a reference story token by token. The prompt supplies context, while the response and EOS token provide the supervised targets.


In [1]:
# Use this version of transformers for consistency.
%pip -q install "transformers==4.57.1" "torch>=2.6,<3"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

torch.set_num_threads(1)
torch.manual_seed(151)
MODEL_ID = "roneneldan/TinyStories-1M"
REVISION = "77f1b168e219585646439073245fe87e56b3023e"

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

## Part 1: Prepare examples and compute loss (50 points)
### 1a: Tokenize examples (15 points)

Load the tokenizer for `MODEL_ID` at `REVISION` with [AutoTokenizer](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/auto#transformers.AutoTokenizer). Set its padding token to its EOS token and its padding side to right.

Implement `encode_example`:

- Make sure to encode the prompt and response **separately** to ensure that token boundaries aren't overlapped between the prompt and response, without automatically added special tokens. See [PreTrainedTokenizer.encode](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/tokenizer#transformers.PreTrainedTokenizer.encode).
- Join the prompt IDs, response IDs, and one EOS ID, in that order.
- Return a dictionary with two Python lists: `input_ids` and `response_mask`. The mask is true at response and EOS positions and false at prompt positions.

Use this separate-encoding convention throughout. Keep the full text in Part 1. The supplied examples fit the model's context window.


**Notes on tokenization**

- Neural networks operate on numerical representations, so a tokenizer maps text into token IDs.
- Encode the prompt and response separately to preserve their boundary, concatenate their IDs, then append one EOS token.
- `AutoTokenizer` loads the tokenizer associated with the pretrained model; EOS indicates the end of the sequence.
- Reusing EOS as the padding token avoids changing the model's vocabulary. The attention mask distinguishes a real EOS position from padding.
- Tokenizer encoding is a text-to-ID preprocessing operation, not the neural-network Transformer encoder.


In [4]:
# Load the tokenizer at the specified model revision.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
# Reuse EOS for right padding without expanding the vocabulary.
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/722 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

In [5]:
def encode_example(prompt, response, tokenizer):
    # Encode separately so the prompt/response boundary is preserved.
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    response_ids = tokenizer.encode(response, add_special_tokens=False)

    # Append exactly one end-of-sequence token.
    input_ids = prompt_ids + response_ids + [tokenizer.eos_token_id]

    # Only response tokens and EOS are supervised targets.
    response_mask = ([False] * len(prompt_ids)
                     + [True] * (len(response_ids) + 1))
    return {"input_ids": input_ids, "response_mask": response_mask}


In [6]:
# Supplied check: this tokenizer assigns one ID to each character.
# "AB" is context and "C" plus EOS are targets.
# This checks the boundary and EOS without depending on subword tokenization.

class CharacterTokenizer:
    eos_token_id = 9
    def encode(self, text, add_special_tokens=False):
        return [ord(c) - ord("A") + 1 for c in text]

assert encode_example("AB", "C", CharacterTokenizer()) == {
    "input_ids": [1, 2, 3, 9],
    "response_mask": [False, False, True, True],
}
assert tokenizer.pad_token_id == tokenizer.eos_token_id
print("Encoding check passed.")


Encoding check passed.


### 1b: Create batches (15 points)

A **collator** combines examples into a batch. Implement `collate_examples` for the dictionaries from 1a. Add padding on the right so all rows match the longest example in the batch.

Return three CPU tensors of shape `(B, T)`, where `B` is the number of examples and `T` is the padded length:

| Field | Type | Contents |
|---|---|---|
| `input_ids` | `torch.long` | Token IDs, with `pad_id` in added positions |
| `attention_mask` | `torch.long` | 1 at real positions, including the prompt; 0 at padding |
| `response_mask` | `torch.bool` | The example's response mask, with false at padding |

It is important to enforce **causality**: each position can use itself and earlier real tokens. Your attention mask identifies real positions. On the other hand, your response mask identifies targets to score.

Create a [DataLoader](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) using your collator and the examples below. Use batch size 2, preserve order, and keep the smaller final batch.


**Notes on batching**

- The collator right-pads variable-length examples to make rectangular `(B, T)` tensors.
- `input_ids` holds token IDs; `attention_mask` is 1 for real tokens (including the prompt) and 0 for padding.
- `response_mask` identifies only the response and EOS targets for the loss; prompt tokens are context, not supervised targets.


In [7]:
# Supplied examples: the request and its reference story stay together.

stories = [
    ("Write a story using moon, fox, boat.\nStory:", " A fox sailed a boat under the moon."),
    ("Use rain, key, garden in a short story.\nStory:", " In the rain, Ada found a key in the garden."),
    ("Write a story using bell, bird, tree.\nStory:", " A bird heard a bell and flew into a tree."),
]
examples = [encode_example(p, r, tokenizer) for p, r in stories]

# The examples are collated into batches below.


In [8]:
def collate_examples(examples, pad_id):
    max_len = max(len(ex["input_ids"]) for ex in examples)
    input_ids, attention_mask, response_mask = [], [], []

    for ex in examples:
        length = len(ex["input_ids"])
        pad_length = max_len - length
        # Pad each sequence on the right to the batch maximum length.
        input_ids.append(ex["input_ids"] + [pad_id] * pad_length)
        # Real tokens, including the prompt and EOS, have attention value 1.
        attention_mask.append([1] * length + [0] * pad_length)
        # Padding never contributes to the response loss.
        response_mask.append(ex["response_mask"] + [False] * pad_length)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "response_mask": torch.tensor(response_mask, dtype=torch.bool),
    }


In [9]:
loader = DataLoader(
    examples,
    batch_size=2,
    shuffle=False,  # Preserve the original example order.
    drop_last=False,  # Keep the smaller final batch.
    collate_fn=lambda batch: collate_examples(batch, tokenizer.pad_token_id),
)


In [10]:
# Supplied check: the shorter row has one padding position.
# EOS and padding share ID 9 but occupy different kinds of positions.
# The loader must retain its final one-example batch.

small = [
    {"input_ids": [1, 2, 9], "response_mask": [False, True, True]},
    {"input_ids": [3, 9], "response_mask": [False, True]},
]
b = collate_examples(small, 9)
assert b["input_ids"].tolist() == [[1, 2, 9], [3, 9, 9]]
assert b["attention_mask"].tolist() == [[1, 1, 1], [1, 1, 0]]
assert b["response_mask"].tolist() == [[False, True, True], [False, True, False]]
assert [len(b["input_ids"]) for b in loader] == [2, 1]
print("Batch check passed.")


Batch check passed.


### 1c: Make labels (10 points)

Implement `make_labels(batch)` so Hugging Face computes the same loss. Return a `(B, T)` long tensor without changing the batch. Read the `labels` parameter documentation for [GPTNeoForCausalLM.forward](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/gpt_neo#transformers.GPTNeoForCausalLM.forward). Determine whether the model shifts labels internally and which label value excludes a position from the loss. Use these conventions to implement `make_labels`.


In [11]:
def make_labels(batch):
    # Clone so creating labels does not alter model inputs.
    labels = batch["input_ids"].clone()
    # Ignore prompt and padding positions; retain response and EOS IDs.
    labels[~batch["response_mask"]] = -100
    # CausalLM shifts the labels internally when labels are provided.
    return labels


In [12]:
# Supplied check: prompt tokens and padding are ignored; response and EOS are targets.
# Each row has two prompt tokens. EOS and padding share ID 9.
# The labels keep the input positions; the model shifts them internally.

label_batch = {
    "input_ids": torch.tensor([[1, 2, 3, 4, 9], [5, 6, 7, 9, 9]]),
    "attention_mask": torch.tensor([[1, 1, 1, 1, 1], [1, 1, 1, 1, 0]]),
    "response_mask": torch.tensor([[False, False, True, True, True],
                                   [False, False, True, True, False]]),
}
label_batch_before = {key: value.clone() for key, value in label_batch.items()}
labels = make_labels(label_batch)
expected_labels = torch.tensor([[-100, -100, 3, 4, 9],
                                [-100, -100, 7, 9, -100]])
assert labels.dtype == torch.long
assert labels.shape == label_batch["input_ids"].shape
torch.testing.assert_close(labels, expected_labels)
for key in label_batch:
    torch.testing.assert_close(label_batch[key], label_batch_before[key])
print("Labels check passed.")


Labels check passed.


### 1d: Compute loss (10 points)

`logits` has shape `(B, T, V)`, where `V` is the vocabulary size. Its vector at position `t` predicts the token at position `t + 1`.

Implement `response_loss(logits, batch)` using [cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html). Score real response tokens and EOS. Return the tuple `(loss_sum, token_count, mean_loss)`:

- `loss_sum`: a scalar tensor containing the sum of scored losses.
- `token_count`: a Python integer counting scored tokens.
- `mean_loss`: a scalar tensor equal to `loss_sum / token_count`.

Keep the loss differentiable.

Load `MODEL_ID` at `REVISION` with [AutoModelForCausalLM](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/auto#transformers.AutoModelForCausalLM) and set evaluation mode. Run the supplied checks. The final cell compares the two losses and prints one example's tokens and masks.


**Notes on logits and next-token prediction**

- For each input position, the model produces a logit for every vocabulary token, yielding a tensor of shape `(B, T, V)`.
- Causal attention prevents each position from using future tokens.
- The logit vector at position `t` predicts the token at `t + 1`, so manual cross-entropy calculation must align shifted logits with shifted targets.


In [13]:
def response_loss(logits, batch):
    labels = make_labels(batch)
    # Position t predicts the label at position t + 1.
    shift_logits = logits[:, :-1, :]
    shift_labels = labels[:, 1:]

    # Sum losses at supervised positions only; keep gradients intact.
    loss_sum = F.cross_entropy(
        shift_logits.reshape(-1, shift_logits.size(-1)),
        shift_labels.reshape(-1),
        ignore_index=-100,
        reduction="sum",
    )
    token_count = int((shift_labels != -100).sum().item())
    mean_loss = loss_sum / token_count
    return loss_sum, token_count, mean_loss


In [14]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION
)
model.eval()



config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/48.6M [00:00<?, ?B/s]

GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 64)
    (wpe): Embedding(2048, 64)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-7): 8 x GPTNeoBlock(
        (ln_1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=64, out_features=64, bias=False)
            (v_proj): Linear(in_features=64, out_features=64, bias=False)
            (q_proj): Linear(in_features=64, out_features=64, bias=False)
            (out_proj): Linear(in_features=64, out_features=64, bias=True)
          )
        )
        (ln_2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=64, out_features=256, bias=True)
          (c_proj): Linear(in_features=256, out_

In [15]:
# Supplied check: uniform predictions over four tokens give log(4) per target.
# There are two supervised targets. Backward must reach the logits.

b = {"input_ids": torch.tensor([[0, 1, 2, 3]]),
     "attention_mask": torch.tensor([[1, 1, 1, 0]]),
     "response_mask": torch.tensor([[False, True, True, False]])}
z = torch.zeros(1, 4, 4, requires_grad=True)
s, n, m = response_loss(z, b)
assert n == 2
assert abs(s.item() - 2 * math.log(4)) < 1e-6
assert abs(m.item() - math.log(4)) < 1e-6
m.backward()
assert z.grad is not None
print("Loss check passed.")


model.safetensors:   0%|          | 0.00/48.6M [00:00<?, ?B/s]

Loss check passed.


In [16]:
# Supplied integration check: both interfaces must score the same targets.
# Print one example's tokens and masks to make their roles visible.

batch = next(iter(loader))
with torch.no_grad():
    output = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    total, count, mean = response_loss(output.logits, batch)
    builtin = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                    labels=make_labels(batch)).loss
assert count == int(batch["response_mask"].sum())
torch.testing.assert_close(mean, builtin, rtol=1e-5, atol=1e-6)
print("Scored tokens:", count, "Mean loss:", mean.item())
print("position | token | attention | response target")
for j, token_id in enumerate(batch["input_ids"][0].tolist()):
    print(j, repr(tokenizer.convert_ids_to_tokens(token_id)),
          int(batch["attention_mask"][0,j]), bool(batch["response_mask"][0,j]))


Scored tokens: 23 Mean loss: 4.988582611083984
position | token | attention | response target
0 'Write' 1 False
1 'Ġa' 1 False
2 'Ġstory' 1 False
3 'Ġusing' 1 False
4 'Ġmoon' 1 False
5 ',' 1 False
6 'Ġfox' 1 False
7 ',' 1 False
8 'Ġboat' 1 False
9 '.' 1 False
10 'Ċ' 1 False
11 'Story' 1 False
12 ':' 1 False
13 'ĠA' 1 True
14 'Ġfox' 1 True
15 'Ġsailed' 1 True
16 'Ġa' 1 True
17 'Ġboat' 1 True
18 'Ġunder' 1 True
19 'Ġthe' 1 True
20 'Ġmoon' 1 True
21 '.' 1 True
22 '<|endoftext|>' 1 True
23 '<|endoftext|>' 0 False
24 '<|endoftext|>' 0 False
25 '<|endoftext|>' 0 False
26 '<|endoftext|>' 0 False


---

## Part 2: Find and fix bugs (50 points)

Each function below passes its supplied example but returns a wrong result for some valid input. These exercises are independent of Part 1. Use small lists or tensors.

For each case:
1. Construct a counterexample. Print your expected result and the buggy result. Determine the expected result from the stated requirements, not from your Part 1 code. A crash alone does not count.
2. Explain the cause in **at most three sentences**.
3. Write a repair named `fixed_...`, leaving `buggy_...` unchanged. Run the supplied assertions; you do not need to write tests.

| Graded work | A | B | C |
|---|---:|---:|---:|
| Counterexample and expected result | 5 | 5 | 8 |
| Diagnosis | 5 | 5 | 5 |
| Repair | 5 | 5 | 7 |


### Case A: Fit an example into a context window (15 points)

The full example may exceed `max_length`. Keep the entire response, append EOS, and keep as many tokens from the **end of the prompt** as will fit. Return the resulting IDs and a mask that is true at response and EOS positions only.

The input lists contain no EOS. The prompt has at least one token, and the response plus EOS leaves room for at least one prompt token. Choose a counterexample where the returned mask differs from the expected mask and both contain at least one true entry.


In [17]:
# Supplied function and ordinary example. Do not edit this function.
# The expected IDs and mask are shown for one input.
def buggy_fit(prompt_ids, response_ids, eos_id, max_length):
    start = len(prompt_ids)
    room = max_length - len(response_ids) - 1
    kept_prompt = prompt_ids[-room:]
    ids = kept_prompt + response_ids + [eos_id]
    mask = [i >= start for i in range(len(ids))]
    return ids, mask

assert buggy_fit([1, 2], [3], 9, 5) == ([1, 2, 3, 9], [False, False, True, True])


In [18]:
# Valid counterexample: both masks contain at least one True value.
# The response plus EOS leaves room for three prompt tokens.
counterexample_a = {
    "prompt_ids": [1, 2, 3, 4],
    "response_ids": [5, 6],
    "eos_id": 9,
    "max_length": 6,
}
expected_a = (
    [2, 3, 4, 5, 6, 9],
    [False, False, False, True, True, True],
)
buggy_result_a = buggy_fit(**counterexample_a)
print("Expected:", expected_a)
print("Buggy:", buggy_result_a)

def fixed_fit(prompt_ids, response_ids, eos_id, max_length):
    room = max_length - len(response_ids) - 1
    if room < 0:
        raise ValueError("Response and EOS exceed max_length")
    kept_prompt = prompt_ids[-room:] if room > 0 else []
    ids = kept_prompt + response_ids + [eos_id]
    start = len(kept_prompt)
    mask = [i >= start for i in range(len(ids))]
    return ids, mask

assert fixed_fit(**counterexample_a) == expected_a
assert fixed_fit([1, 2], [3], 9, 5) == ([1, 2, 3, 9], [False, False, True, True])


Expected: ([2, 3, 4, 5, 6, 9], [False, False, False, True, True, True])
Buggy: ([2, 3, 4, 5, 6, 9], [False, False, False, False, True, True])


<!-- BEGIN QUESTION -->

### Answer to Case A

In [19]:
print("Expected:", expected_a)
print("Buggy:", buggy_result_a)

Expected: ([2, 3, 4, 5, 6, 9], [False, False, False, True, True, True])
Buggy: ([2, 3, 4, 5, 6, 9], [False, False, False, False, True, True])


**Your diagnosis (at most three sentences):** The function records the original prompt length before truncation, then incorrectly uses it as the response start index in the shortened sequence. Consequently, the first response target is marked as part of the prompt, even though it must be scored. The repair computes the mask boundary from the retained prompt length.


<!-- END QUESTION -->

**End of written response — do not edit, delete, or move this cell.**

Keep your entire answer, including supporting tables and figures, above this boundary.

### Case B: Score response predictions (15 points)

Return the mean next-token loss over response tokens, including EOS. `logits` has shape `(B, T, V)`. `input_ids` and the boolean `response_mask` have shape `(B, T)`. Position `t` predicts token `t + 1`.

Each sequence has at least one prompt token followed by at least two response targets, counting EOS. There is no padding in this exercise. Construct a counterexample with one sequence, at most four positions, and at most four vocabulary entries. Choose the token IDs, mask, and logits yourself. Show the expected mean from the probabilities of the scored targets.


In [20]:
# Supplied function and ordinary example. Do not edit this function.
# The expected mean for this input is log(3).
def buggy_response_loss(logits, input_ids, response_mask):
    predictions = logits[:, :-1, :]
    targets = input_ids[:, 1:]
    selected = response_mask[:, :-1]
    losses = F.cross_entropy(
        predictions.reshape(-1, logits.shape[-1]),
        targets.reshape(-1), reduction="none",
    ).reshape_as(targets)
    return losses[selected].mean()

ordinary_b = (torch.zeros(1, 3, 3), torch.tensor([[0, 1, 2]]),
              torch.tensor([[False, True, True]]))
assert abs(buggy_response_loss(*ordinary_b).item() - math.log(3)) < 1e-6


In [21]:
# One sequence (T=4) and three vocabulary entries.
# The scored next tokens are token 2 and token 0 (EOS in this toy case).
counterexample_b = {
    "logits": torch.tensor([[
        [0.0, 0.0, 0.0],
        [0.0, 0.0, math.log(4.0)],
        [0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0],
    ]]),
    "input_ids": torch.tensor([[0, 1, 2, 0]]),
    "response_mask": torch.tensor([[False, False, True, True]]),
}
# For target 2, p=4/(1+1+4)=2/3; for EOS target 0, p=1/3.
expected_b = (-math.log(2.0 / 3.0) - math.log(1.0 / 3.0)) / 2.0
buggy_result_b = buggy_response_loss(**counterexample_b)
print("Expected:", expected_b)
print("Buggy:", buggy_result_b.item())

def fixed_response_loss(logits, input_ids, response_mask):
    predictions = logits[:, :-1, :]
    targets = input_ids[:, 1:]
    # The selection mask must refer to target positions, not prediction positions.
    selected = response_mask[:, 1:]
    losses = F.cross_entropy(
        predictions.reshape(-1, logits.shape[-1]),
        targets.reshape(-1), reduction="none",
    ).reshape_as(targets)
    mean_loss = losses[selected].mean()
    return mean_loss

assert math.isclose(fixed_response_loss(**counterexample_b).item(), expected_b, rel_tol=1e-6, abs_tol=1e-6)
assert math.isclose(fixed_response_loss(*ordinary_b).item(), math.log(3), rel_tol=1e-6, abs_tol=1e-6)


Expected: 0.7520386983881371
Buggy: 1.0986123085021973


<!-- BEGIN QUESTION -->

### Answer to Case B

In [22]:
print("Expected:", expected_b)
print("Buggy:", buggy_result_b.item())

Expected: 0.7520386983881371
Buggy: 1.0986123085021973


**Your diagnosis (at most three sentences):** Logits at position `t` predict the token at `t + 1`, so the response mask must be aligned with the shifted targets. The buggy function drops the last mask element instead of the first, excluding the first response target from the loss. The repair uses `response_mask[:, 1:]` to select the correct target positions.


<!-- END QUESTION -->

**End of written response — do not edit, delete, or move this cell.**

Keep your entire answer, including supporting tables and figures, above this boundary.

### Case C: Combine losses (20 points)

Each row of `token_losses` contains one example's next-token losses. `score_mask` marks the losses to include. Return the mean over **all scored tokens**. Unscored entries must not affect it.

Every row has at least one scored token. Construct a counterexample with at most two rows and three columns. Then give an unequal-count example where the function is correct. Show both results, run the supplied assertions, and explain the accidental success within your three-sentence diagnosis. No model is needed.


In [23]:
# Supplied function and ordinary example. Do not edit this function.
# The four scored losses below have mean 2.5.
def buggy_mean(token_losses, score_mask):
    totals = (token_losses * score_mask).sum(dim=1)
    counts = score_mask.sum(dim=1)
    return (totals / counts).mean()

ordinary_c = (torch.tensor([[1.,3.],[2.,4.]]), torch.ones(2,2,dtype=torch.bool))
assert buggy_mean(*ordinary_c).item() == 2.5


In [24]:
# Different scored token counts and row means reveal the bug.
counterexample_c = {
    "token_losses": torch.tensor([[1., 3., 100.], [2., 4., 6.]]),
    "score_mask": torch.tensor([[True, True, False], [True, True, True]]),
}
expected_c = (1.0 + 3.0 + 2.0 + 4.0 + 6.0) / 5.0
buggy_result_c = buggy_mean(**counterexample_c)
print("Expected:", expected_c)
print("Buggy:", buggy_result_c.item())

# Despite unequal token counts, equal row means make the buggy method agree.
unequal_count_example_c = {
    "token_losses": torch.tensor([[1., 3., 100.], [2., 2., 2.]]),
    "score_mask": torch.tensor([[True, True, False], [True, True, True]]),
}
expected_unequal_c = 2.0
buggy_unequal_c = buggy_mean(**unequal_count_example_c)
print("Expected (unequal counts):", expected_unequal_c)
print("Buggy (unequal counts):", buggy_unequal_c.item())
assert math.isclose(buggy_unequal_c.item(), expected_unequal_c, rel_tol=1e-6, abs_tol=1e-6)

def fixed_mean(token_losses, score_mask):
    # Weight each scored token equally, not each sequence equally.
    total_loss = (token_losses * score_mask).sum()
    total_count = score_mask.sum()
    mean_loss = total_loss / total_count
    return mean_loss

assert math.isclose(fixed_mean(**counterexample_c).item(), expected_c, rel_tol=1e-6, abs_tol=1e-6)
assert math.isclose(fixed_mean(**unequal_count_example_c).item(), expected_unequal_c, rel_tol=1e-6, abs_tol=1e-6)
assert math.isclose(fixed_mean(*ordinary_c).item(), 2.5, rel_tol=1e-6, abs_tol=1e-6)


Expected: 3.2
Buggy: 3.0
Expected (unequal counts): 2.0
Buggy (unequal counts): 2.0


<!-- BEGIN QUESTION -->

### Answer to Case C

In [25]:
print("Expected:", expected_c)
print("Buggy:", buggy_result_c.item())
print("Expected (unequal counts):", expected_unequal_c)
print("Buggy (unequal counts):", buggy_unequal_c.item())

Expected: 3.2
Buggy: 3.0
Expected (unequal counts): 2.0
Buggy (unequal counts): 2.0


**Your diagnosis (at most three sentences):** The function averages per-sequence means, giving each sequence equal weight even when their scored token counts differ. The required mean divides the total scored loss across the batch by the total number of scored tokens. With unequal counts, the buggy function can still be correct by coincidence when the per-sequence means are equal.


<!-- END QUESTION -->

**End of written response — do not edit, delete, or move this cell.**

Keep your entire answer, including supporting tables and figures, above this boundary.